In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval
except Exception:
    pass

In [2]:
import logging

import numpy as np
import polars as pl

from dataeval import Metadata
from dataeval.config import set_max_processes
from dataeval.core import compute_stats
from dataeval.flags import ImageStats
from dataeval.protocols import DatasetMetadata, DatumMetadata
from dataeval.utils.preprocessing import ChannelGroup

# Statistics are calculated across a process pool. Capping it keeps this example's memory
# footprint predictable; leave it unset to use every available core.
set_max_processes(4)

# Several points in this guide are reported through DataEval's logger rather than raised.
# Turning it on is what makes them visible here; see
# [how to configure logging](./h2_configure_logging.py) for the full picture.
logging.basicConfig(format="%(levelname)s: %(message)s", force=True)
logging.getLogger("dataeval").setLevel(logging.WARNING)

In [3]:
rng = np.random.default_rng(0)


class BandCubes:
    """A small image-classification dataset of four-band cubes: R, G, B, NIR."""

    index2label = {0: "bare", 1: "vegetated"}

    def __init__(self, count: int = 24) -> None:
        self._labels = [i % 2 for i in range(count)]
        self._cubes = []
        for label in self._labels:
            cube = np.empty((4, 32, 32), np.uint16)
            # Visible bands are the same scene either way — 8-bit values.
            cube[:3] = rng.integers(40, 200, (3, 32, 32))
            # Near-infrared separates them, on a 12-bit sensor's scale.
            level = 3000 if label else 500
            cube[3] = rng.integers(level, level + 800, (32, 32))
            self._cubes.append(cube)
        self.metadata = DatasetMetadata(id="band-cubes", index2label=self.index2label)

    def __len__(self) -> int:
        return len(self._cubes)

    def __getitem__(self, i: int) -> tuple[np.ndarray, np.ndarray, DatumMetadata]:
        return self._cubes[i], np.eye(2)[self._labels[i]], DatumMetadata(id=i)


dataset = BandCubes()
print(f"{len(dataset)} scenes, each {dataset[0][0].shape} of dtype {dataset[0][0].dtype}")

24 scenes, each (4, 32, 32) of dtype uint16


In [4]:
stats = ImageStats.PIXEL_MEAN | ImageStats.VISUAL_BRIGHTNESS | ImageStats.DIMENSION_DEPTH

whole = compute_stats(dataset, stats=stats, normalize_pixel_values=False)

for name, values in sorted(whole["stats"].items()):
    print(f"{name:12} first scene: {float(values[0]):10.2f}")

brightness   first scene:       5.92
depth        first scene:      12.00
mean         first scene:     312.36


In [5]:
grouped = compute_stats(
    dataset,
    stats=stats,
    normalize_pixel_values=False,
    channels={"rgb": [0, 1, 2], "nir": 3},
)

for name, values in sorted(grouped["stats"].items()):
    print(f"{name:16} first scene: {float(values[0]):10.2f}")

brightness       first scene:       5.92
depth            first scene:      12.00
mean             first scene:     312.36
nir_brightness   first scene:      42.83
nir_depth        first scene:      12.00
nir_mean         first scene:     885.79
rgb_brightness   first scene:      81.00
rgb_depth        first scene:       8.00
rgb_mean         first scene:     121.21


In [6]:
labels = np.array([dataset[i][1].argmax() for i in range(len(dataset))])
comparison = pl.DataFrame({
    "class": [dataset.index2label[int(label)] for label in labels],
    "nir_mean": np.asarray(grouped["stats"]["nir_mean"], dtype=float),
    "rgb_mean": np.asarray(grouped["stats"]["rgb_mean"], dtype=float),
    "whole_mean": np.asarray(grouped["stats"]["mean"], dtype=float),
})

print(comparison.group_by("class").agg(pl.all().mean().round(1)).sort("class"))

shape: (2, 4)
┌───────────┬──────────┬──────────┬────────────┐
│ class     ┆ nir_mean ┆ rgb_mean ┆ whole_mean │
│ ---       ┆ ---      ┆ ---      ┆ ---        │
│ str       ┆ f64      ┆ f64      ┆ f64        │
╞═══════════╪══════════╪══════════╪════════════╡
│ bare      ┆ 902.2    ┆ 119.5    ┆ 315.2      │
│ vegetated ┆ 3398.5   ┆ 119.1    ┆ 939.0      │
└───────────┴──────────┴──────────┴────────────┘


In [7]:
metadata = Metadata(dataset)
metadata.add_factors(grouped)

print("factors:", sorted(metadata.factor_names))

factors: ['brightness', 'depth', 'id', 'mean', 'nir_brightness', 'nir_depth', 'nir_mean', 'rgb_brightness', 'rgb_depth', 'rgb_mean']


In [8]:
rows = metadata.rows_at("instance")
print(
    rows
    .group_by("class_label")
    .agg(
        pl.col("nir_mean").mean().round(1).alias("nir"),
        pl.col("rgb_mean").mean().round(1).alias("rgb"),
        pl.len().alias("scenes"),
    )
    .sort("class_label")
)

shape: (2, 4)
┌─────────────┬────────────┬────────────┬────────┐
│ class_label ┆ nir        ┆ rgb        ┆ scenes │
│ ---         ┆ ---        ┆ ---        ┆ ---    │
│ i64         ┆ f32        ┆ f32        ┆ u32    │
╞═════════════╪════════════╪════════════╪════════╡
│ 0           ┆ 902.200012 ┆ 119.5      ┆ 12     │
│ 1           ┆ 3398.5     ┆ 119.099998 ┆ 12     │
└─────────────┴────────────┴────────────┴────────┘


In [9]:
ragged = [dataset[0][0], dataset[1][0][:3]]

patchy = compute_stats(
    ragged,
    stats=ImageStats.PIXEL_MEAN | ImageStats.PIXEL_MISSING,
    normalize_pixel_values=False,
    channels={"nir": 3},
)["stats"]

print(pl.DataFrame({name: np.asarray(values, dtype=float) for name, values in sorted(patchy.items())}))

shape: (2, 4)
┌────────────┬─────────┬────────────┬─────────────┐
│ mean       ┆ missing ┆ nir_mean   ┆ nir_missing │
│ ---        ┆ ---     ┆ ---        ┆ ---         │
│ f64        ┆ f64     ┆ f64        ┆ f64         │
╞════════════╪═════════╪════════════╪═════════════╡
│ 312.357666 ┆ 0.0     ┆ 885.792969 ┆ 0.0         │
│ 118.021812 ┆ 0.0     ┆ NaN        ┆ 1.0         │
└────────────┴─────────┴────────────┴─────────────┘


In [10]:
boxed = [(cube, [(4, 4, 16, 16)]) for cube in (dataset[0][0], dataset[1][0])]

composed = compute_stats(
    [image for image, _ in boxed],
    boxes=[boxes for _, boxes in boxed],
    stats=ImageStats.PIXEL_MEAN,
    per_target=False,
    per_background=True,
    normalize_pixel_values=False,
    channels={"nir": 3},
)["stats"]

print(sorted(composed))

['background_fraction', 'background_mean', 'background_nir_mean', 'mean', 'nir_mean']


In [11]:
physical = np.stack([
    rng.normal(0.0, 500.0, (32, 32)),  # elevation, metres relative to sea level
    rng.normal(0.0, 50.0, (32, 32)),  # a tighter instrument on the same scene
])

declared = compute_stats(
    [physical],
    stats=ImageStats.PIXEL_ENTROPY,
    normalize_pixel_values=False,
    channels={
        "elevation": ChannelGroup(0, value_range=(-2000.0, 2000.0)),
        "instrument": ChannelGroup(1, value_range=(-200.0, 200.0)),
    },
)["stats"]

for name, values in sorted(declared.items()):
    print(f"{name:20} {float(values[0]):.3f}")

elevation_entropy    4.807
entropy              nan
instrument_entropy   4.778


In [12]:
geometry = compute_stats(
    dataset,
    stats=ImageStats.DIMENSION_WIDTH | ImageStats.DIMENSION_CHANNELS | ImageStats.DIMENSION_DEPTH,
    normalize_pixel_values=False,
    channels={"rgb": [0, 1, 2], "nir": 3},
)["stats"]

print(sorted(geometry))

['channels', 'depth', 'nir_depth', 'rgb_depth', 'width']
